# H1 외부검증: 시간 분할 + 접속 활동

**목적:** 성장 정체 군집을 곧바로 주차 유저로 해석하지 않고, 클러스터링에 사용하지 않은 후반기 성장 정체 지속성과 접속 활동으로 **주차 후보**를 선별한다.

공개 API에는 주간 보스 수행 기록과 메소 생산량이 없다. 따라서 이 노트북의 결론은 `주차 유저 확정`이 아니라 `주차 후보 선별`이다.

이 노트북은 두 모드를 지원한다.

1. **예비 모드:** 현재 저장된 집계 CSV로 기존 성장 정체 군집을 재현하고 최근 6개월 정체 지속성과 최신 접속 여부를 진단한다.
2. **완전 시간 분할 모드:** 월별 `access_flag`, `union_level`이 원시 CSV에 있으면 전반 6개월로 군집을 만들고 후반 6개월로 외부검증한다.


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
K = 3
DATA = Path('../data')
MONTHS = [f'2025-{m:02d}' for m in range(6, 13)] + [f'2026-{m:02d}' for m in range(1, 6)]
EARLY_MONTHS, LATE_MONTHS = MONTHS[:6], MONTHS[6:]

features = pd.read_csv(DATA / 'features_monthly.csv', encoding='utf-8-sig')
main = pd.read_csv(DATA / 'main_characters.csv', encoding='utf-8-sig', usecols=['ocid', 'class_group'])
hexa = pd.read_csv(DATA / 'hexa_fragments.csv', encoding='utf-8-sig')
raw = pd.read_csv(DATA / 'monthly_snapshots_raw.csv', encoding='utf-8-sig')

required_raw = {'ocid', 'year_month', 'level', 'exp', 'union_level', 'access_flag'}
missing_raw = sorted(required_raw - set(raw.columns))
full_temporal_ready = not missing_raw

print(f'features: {features.shape} | hexa: {hexa.shape} | raw: {raw.shape}')
print(f'전반기: {EARLY_MONTHS[0]} ~ {EARLY_MONTHS[-1]} | 후반기: {LATE_MONTHS[0]} ~ {LATE_MONTHS[-1]}')
print(f'완전 시간 분할 모드 가능: {full_temporal_ready}')
if missing_raw:
    print('현재 raw CSV 누락 필드:', missing_raw)
    print('→ 기존 CSV에서는 예비 모드를 실행한다. scripts/collect_features.py 재수집 후 완전 모드가 자동 활성화된다.')


features: (2000, 44) | hexa: (2000, 23) | raw: (24000, 12)
전반기: 2025-06 ~ 2025-11 | 후반기: 2025-12 ~ 2026-05
완전 시간 분할 모드 가능: True


## 1. 현재 CSV 기반 예비 외부검증

기존 H1 성장 정체 군집은 12개월 전체 집계 피처로 재현한다. 검증 신호는 최근 6개월 변화량과 최신 접속 여부다.

> 전체 기간 집계가 군집 생성에 사용되므로 엄밀한 시간 분할 검증은 아니다. 이 결과는 현재 라벨의 해석 가능성을 점검하는 진단이다.


In [2]:
df = (features
      .merge(main, on='ocid', how='left')
      .merge(hexa[['ocid', 'avg_monthly_delta_hexa_frag', 'recent6_delta_hexa_frag']], on='ocid', how='left'))
df = df[df['level'].between(270, 290)].copy()
# 분석 표본 분모 = h1_clustering.ipynb df_final 동일 정의(핵심 delta dropna) → 비율 통일
n_analysis = len(df.dropna(subset=['avg_monthly_delta_level',
                                   'avg_monthly_delta_combat_power',
                                   'avg_monthly_delta_union_level']))
df = df.dropna(subset=['log1p_avg_monthly_delta_cumexp',
                       'avg_monthly_delta_union_level',
                       'access_ratio',
                       'avg_monthly_delta_hexa_frag']).copy()

df['cumexp_avg'] = df['log1p_avg_monthly_delta_cumexp'].clip(lower=0)
df['union_avg'] = df['avg_monthly_delta_union_level'].clip(lower=0)
df['hfrag_avg'] = df['avg_monthly_delta_hexa_frag'].clip(lower=0)
df['access_ratio_feature'] = df['access_ratio'].fillna(0)
cluster_features = ['cumexp_avg', 'union_avg', 'access_ratio_feature']

X = StandardScaler().fit_transform(df[cluster_features])
km = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE).fit(X)
df['cluster'] = km.labels_
stagnant_cluster_id = int(df.groupby('cluster')['cumexp_avg'].mean().idxmin())
df['is_stagnant_cluster'] = df['cluster'].eq(stagnant_cluster_id)

df['late_stagnant'] = (
    df['log1p_recent6_delta_cumexp'].fillna(np.inf).le(0)
    & df['recent6_delta_union_level'].fillna(np.inf).le(0)
    & df['recent6_delta_hexa_frag'].fillna(np.inf).le(0)
)
df['recent_access'] = df['access_recent'].eq(1)
df['access_ratio_ge_50'] = df['access_ratio'].ge(0.5)
df['likely_candidate_prelim'] = df['is_stagnant_cluster'] & df['late_stagnant'] & df['recent_access']

profile = df.groupby('cluster').agg(
    n=('ocid', 'size'),
    cumexp_avg=('cumexp_avg', 'mean'),
    union_avg=('union_avg', 'mean'),
    hfrag_avg=('hfrag_avg', 'mean'),
    access_ratio_feature=('access_ratio_feature', 'mean'),
    late_stagnant_rate=('late_stagnant', 'mean'),
    recent_access_rate=('recent_access', 'mean'),
    access_ratio_mean=('access_ratio', 'mean'),
    likely_candidate_prelim_n=('likely_candidate_prelim', 'sum'),
)

print(f'K-Means silhouette: {silhouette_score(X, km.labels_):.4f}')
print(f'성장 정체 클러스터 ID: {stagnant_cluster_id}')
display(profile.round(3))

st = df[df['is_stagnant_cluster']]
print('\n[예비 판정]')
print(f'성장 정체 클러스터: {len(st):,}명 ({len(st)/n_analysis:.1%})')
print(f'후반기 6개월 정체 지속: {st["late_stagnant"].mean():.1%}')
print(f'최신 접속 관측: {st["recent_access"].mean():.1%}')
print(f'전체 기간 access_ratio 평균: {st["access_ratio"].mean():.1%}')
print(f'예비 주차 후보(정체 군집 AND 후반기 정체 AND 최신 접속): {st["likely_candidate_prelim"].sum():,}명')


K-Means silhouette: 0.5437
성장 정체 클러스터 ID: 1


,n,cumexp_avg,union_avg,hfrag_avg,access_ratio_feature,late_stagnant_rate,recent_access_rate,access_ratio_mean,likely_candidate_prelim_n
cluster,,,,,,,,,
0,1436,30.138,41.410,184.709,0.935,0.003,1.0,0.935,0
1,417,27.044,38.940,37.529,0.487,0.089,1.0,0.487,37
2,126,30.047,380.812,185.348,0.868,0.008,1.0,0.868,0



[예비 판정]
성장 정체 클러스터: 417명 (21.1%)
후반기 6개월 정체 지속: 8.9%
최신 접속 관측: 100.0%
전체 기간 access_ratio 평균: 48.7%
예비 주차 후보(정체 군집 AND 후반기 정체 AND 최신 접속): 37명


## 2. 재수집 후 자동 실행되는 완전 시간 분할 검증

다음 셀은 월별 원시 CSV에 접속·유니온 필드가 보존되면 자동으로 실행된다. 전반기 피처만으로 군집을 생성하고 후반기 정체 지속과 반복 접속을 검증한다.


In [3]:
def avg_delta(group, col):
    g = group[['month_idx', col]].dropna().sort_values('month_idx')
    if len(g) < 2 or g['month_idx'].iloc[-1] == g['month_idx'].iloc[0]:
        return np.nan
    return (g[col].iloc[-1] - g[col].iloc[0]) / (g['month_idx'].iloc[-1] - g['month_idx'].iloc[0])

def add_cumexp(raw_df):
    req = pd.read_csv(DATA / 'exp_requirement_table.csv', encoding='utf-8-sig')
    req_map = req.set_index('level')['requirement'].to_dict()
    base = {270: 0.0}
    for level in range(271, 291):
        base[level] = base[level - 1] + req_map[level - 1]
    out = raw_df.copy()
    out = out[out['level'].between(270, 290)].copy()
    out['cumexp'] = out.apply(lambda r: base[int(r['level'])] + float(r['exp']), axis=1)
    return out

def build_window(raw_df, months, prefix):
    part = raw_df[raw_df['year_month'].isin(months)].copy()
    rows = []
    for ocid, g in part.groupby('ocid'):
        rows.append({
            'ocid': ocid,
            f'{prefix}_cumexp': np.log1p(max(0, avg_delta(g, 'cumexp'))),
            f'{prefix}_union': max(0, avg_delta(g, 'union_level')),
            f'{prefix}_hfrag': max(0, avg_delta(g, 'hexa_frag')),
            f'{prefix}_access_active_months': g['access_flag'].fillna(0).sum(),
            f'{prefix}_access_ratio': g['access_flag'].mean(),
        })
    return pd.DataFrame(rows)

if not full_temporal_ready:
    print('SKIP: 현재 raw CSV에는 월별 access_flag / union_level이 없다.')
    print('scripts/collect_features.py 보완 버전으로 재수집하면 이 셀이 자동 실행된다.')
else:
    frag_cols = [c for c in hexa.columns if c.endswith('_hexa_frag') and c[:7] in MONTHS]
    frag_long = hexa[['ocid', *frag_cols]].melt(id_vars='ocid', var_name='frag_month', value_name='hexa_frag')
    frag_long['year_month'] = frag_long['frag_month'].str[:7]
    monthly = raw.merge(frag_long[['ocid', 'year_month', 'hexa_frag']], on=['ocid', 'year_month'], how='left')
    monthly['month_idx'] = monthly['year_month'].map({m: i for i, m in enumerate(MONTHS)})
    monthly = add_cumexp(monthly)

    early = build_window(monthly, EARLY_MONTHS, 'early')
    late = build_window(monthly, LATE_MONTHS, 'late')
    split = early.merge(late, on='ocid', how='inner').dropna().copy()
    early_cols = ['early_cumexp', 'early_union', 'early_access_ratio']
    X_early = StandardScaler().fit_transform(split[early_cols])
    km_early = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE).fit(X_early)
    split['early_cluster'] = km_early.labels_
    early_stagnant_id = int(split.groupby('early_cluster')['early_cumexp'].mean().idxmin())
    split['early_stagnant_cluster'] = split['early_cluster'].eq(early_stagnant_id)
    split['late_stagnant'] = split['late_cumexp'].le(0) & split['late_union'].le(0) & split['late_hfrag'].le(0)
    split['late_repeated_access'] = split['late_access_active_months'].ge(4)
    split['likely_candidate'] = split['early_stagnant_cluster'] & split['late_stagnant'] & split['late_repeated_access']

    print(f'전반기 학습 silhouette: {silhouette_score(X_early, km_early.labels_):.4f}')
    print(f'전반기 성장 정체 클러스터: {split["early_stagnant_cluster"].sum():,}명')
    print(f'후반기 정체 지속 + 반복 접속 주차 후보: {split["likely_candidate"].sum():,}명')
    display(split.groupby('early_cluster').agg(
        n=('ocid', 'size'),
        late_stagnant_rate=('late_stagnant', 'mean'),
        late_repeated_access_rate=('late_repeated_access', 'mean'),
        likely_candidate_n=('likely_candidate', 'sum'),
    ).round(3))


전반기 학습 silhouette: 0.7219
전반기 성장 정체 클러스터: 202명
후반기 정체 지속 + 반복 접속 주차 후보: 11명


,n,late_stagnant_rate,late_repeated_access_rate,likely_candidate_n
early_cluster,,,,
0,78,0.013,0.949,0
1,1517,0.015,0.974,0
2,202,0.089,0.787,11


## 3. 완전 시간 분할 기준 민감도

후반기 성장 정체 축 개수와 반복 접속 개월 수 임계값을 바꿔 주차 후보 수가 어떻게 달라지는지 확인한다. 최종 권고 기준은 `3개 축 모두 정체 AND 4개월 이상 접속`이다.


In [4]:
if not full_temporal_ready:
    print('SKIP: 완전 시간 분할 모드가 필요하다.')
else:
    split['late_zero_axes'] = (
        split['late_cumexp'].le(0).astype(int)
        + split['late_union'].le(0).astype(int)
        + split['late_hfrag'].le(0).astype(int)
    )
    sensitivity = []
    for min_axes in [2, 3]:
        for min_access_months in [1, 2, 3, 4]:
            mask = (
                split['early_stagnant_cluster']
                & split['late_zero_axes'].ge(min_axes)
                & split['late_access_active_months'].ge(min_access_months)
            )
            sensitivity.append({
                'late_zero_axes_min': min_axes,
                'late_access_months_min': min_access_months,
                'candidate_n': int(mask.sum()),
            })
    sensitivity = pd.DataFrame(sensitivity)
    print('[전반기 성장 정체 군집 내부 후보 수 민감도]')
    display(sensitivity.pivot(index='late_zero_axes_min', columns='late_access_months_min', values='candidate_n'))
    print('\n[전반기 성장 정체 군집: 후반기 정체 축 수 × 접속 활성 개월 수]')
    display(pd.crosstab(
        split.loc[split['early_stagnant_cluster'], 'late_zero_axes'],
        split.loc[split['early_stagnant_cluster'], 'late_access_active_months'],
    ))


[전반기 성장 정체 군집 내부 후보 수 민감도]


late_access_months_min,1,2,3,4
late_zero_axes_min,,,,
2,52,52,43,30
3,18,18,15,11



[전반기 성장 정체 군집: 후반기 정체 축 수 × 접속 활성 개월 수]


late_access_active_months,2.0,3.0,4.0,5.0,6.0
late_zero_axes,,,,,
0,2,8,24,19,49
1,3,8,10,10,17
2,6,9,9,5,5
3,3,4,2,2,7


## 해석 원칙

- `성장 정체 군집`: 성장량 피처만으로 분리된 군집. 휴면 캐릭터가 포함될 수 있다.
- `주차 후보`: 성장 정체 군집에 속하고, 후반기에도 정체가 지속되며, 접속 활동이 반복 관측된 캐릭터.
- `주차 유저`: 공개 API만으로 확정할 수 없다. 주간 보스 수행과 메소 생산 기록이 없기 때문이다.


## 4. Repeated-access candidate concentration

A stagnant-growth cluster is not a confirmed candidate label. This section checks whether late-window candidates are concentrated in the early stagnant-growth cluster with a one-sided Fisher exact test.

- Strict: all three growth axes stagnant and access observed in at least four late months
- Relaxed: at least two growth axes stagnant and access observed in at least four late months

In [5]:
from scipy.stats import fisher_exact

if not full_temporal_ready:
    print('SKIP: full temporal data is required.')
else:
    split['late_zero_axes'] = (
        split['late_cumexp'].le(0).astype(int)
        + split['late_union'].le(0).astype(int)
        + split['late_hfrag'].le(0).astype(int)
    )
    rows = []
    for label, min_axes in [('strict', 3), ('relaxed', 2)]:
        candidate = split['late_zero_axes'].ge(min_axes) & split['late_access_active_months'].ge(4)
        table = pd.crosstab(split['early_stagnant_cluster'], candidate).reindex(
            index=[False, True], columns=[False, True], fill_value=0
        )
        odds_ratio, p_value = fisher_exact(table.values, alternative='greater')
        rows.append({
            'criterion': label,
            'late_candidate_n': int(candidate.sum()),
            'captured_in_early_stagnant_cluster': int((candidate & split['early_stagnant_cluster']).sum()),
            'capture_rate': float(split.loc[candidate, 'early_stagnant_cluster'].mean()) if candidate.any() else np.nan,
            'odds_ratio': float(odds_ratio),
            'fisher_p_one_sided': float(p_value),
        })
    concentration = pd.DataFrame(rows)
    display(concentration.round(4))
    print('Interpretation: test candidate concentration inside the stagnant-growth cluster.')

,criterion,late_candidate_n,captured_in_early_stagnant_cluster,capture_rate,odds_ratio,fisher_p_one_sided
0,strict,20,11,0.5500,10.1489,0.0
1,relaxed,74,30,0.4054,6.1483,0.0


Interpretation: test candidate concentration inside the stagnant-growth cluster.


## 5. Current-oriented quarterly external validation

A fixed six-month state is sensitive to updates and seasonal changes. This section evaluates whether the previous quarter stagnant-growth cluster captures strict candidates in the next quarter.

- Strict candidate: all three growth axes stagnant
- Activity gate: access observed in at least two of three validation months
- Output: `data/h1_current_candidates.csv`

In [6]:
# ROLLING_QUARTERLY_VALIDATION
from scipy.stats import fisher_exact

if not full_temporal_ready:
    print('SKIP: full temporal data is required.')
else:
    quarterly_rows = []
    latest_labels = None
    for start in [0, 3, 6]:
        train_months = MONTHS[start:start + 3]
        valid_months = MONTHS[start + 3:start + 6]
        train = build_window(monthly, train_months, 'train')
        valid = build_window(monthly, valid_months, 'valid')
        q = train.merge(valid, on='ocid', how='inner').dropna().copy()
        train_cols = ['train_cumexp', 'train_union', 'train_access_ratio']
        Xq = StandardScaler().fit_transform(q[train_cols])
        kmq = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE).fit(Xq)
        stagnant_id = int(q.assign(_cluster=kmq.labels_).groupby('_cluster')['train_cumexp'].mean().idxmin())
        q['prior_stagnant_cluster'] = kmq.labels_ == stagnant_id
        q['current_zero_axes'] = (
            q['valid_cumexp'].le(0).astype(int)
            + q['valid_union'].le(0).astype(int)
            + q['valid_hfrag'].le(0).astype(int)
        )
        q['is_current_parking_candidate'] = q['current_zero_axes'].eq(3) & q['valid_access_active_months'].ge(2)
        q['is_high_confidence_candidate'] = q['prior_stagnant_cluster'] & q['is_current_parking_candidate']
        table = pd.crosstab(q['prior_stagnant_cluster'], q['is_current_parking_candidate']).reindex(
            index=[False, True], columns=[False, True], fill_value=0
        )
        odds_ratio, p_value = fisher_exact(table.values, alternative='greater')
        quarterly_rows.append({
            'train_window': f'{train_months[0]} ~ {train_months[-1]}',
            'validation_window': f'{valid_months[0]} ~ {valid_months[-1]}',
            'train_silhouette': silhouette_score(Xq, kmq.labels_),
            'current_candidate_n': int(q['is_current_parking_candidate'].sum()),
            'captured_n': int(q['is_high_confidence_candidate'].sum()),
            'capture_rate': float(q.loc[q['is_current_parking_candidate'], 'prior_stagnant_cluster'].mean()),
            'odds_ratio': float(odds_ratio),
            'fisher_p_one_sided': float(p_value),
        })
        if start == 6:
            latest_labels = q[['ocid', 'prior_stagnant_cluster', 'current_zero_axes',
                               'valid_access_active_months', 'is_current_parking_candidate',
                               'is_high_confidence_candidate']].copy()

    quarterly_validation = pd.DataFrame(quarterly_rows)
    display(quarterly_validation.round(4))
    latest_labels.to_csv(DATA / 'h1_current_candidates.csv', index=False, encoding='utf-8-sig')
    print('saved:', DATA / 'h1_current_candidates.csv', latest_labels.shape)
    print('Interpretation: latest quarter is significant, but earlier-quarter consistency is limited.')

,train_window,validation_window,train_silhouette,current_candidate_n,captured_n,capture_rate,odds_ratio,fisher_p_one_sided
0,2025-06 ~ 2025-08,2025-09 ~ 2025-11,0.8015,40,13,0.3250,5.5388,0.0
1,2025-09 ~ 2025-11,2025-12 ~ 2026-02,0.7673,55,32,0.5818,12.3016,0.0
2,2025-12 ~ 2026-02,2026-03 ~ 2026-05,0.7286,63,35,0.5556,9.5379,0.0


saved: ..\data\h1_current_candidates.csv (1884, 6)
Interpretation: latest quarter is significant, but earlier-quarter consistency is limited.
